In [1]:
import time
from lakeshore import Model240

PORT = "COM4"   # change if needed


def inspect_channel(device, ch):
    try:
        temp = device.query(f"KRDG? {ch}").strip()
    except Exception:
        temp = "ERROR"

    try:
        sensor = device.query(f"SRDG? {ch}").strip()
    except Exception:
        sensor = "ERROR"

    print(f"CH{ch}: Temp = {temp}, Sensor = {sensor}")

In [2]:
def scan_all_channels(device):
    print("=== Channel Scan ===")
    for ch in range(1, 9):  # Model240 has up to 8
        inspect_channel(device, ch)

In [3]:
def inspect_config(device, ch):
    try:
        cfg = device.get_input_parameter(ch)
        print(f"CH{ch} Config: {cfg}")
    except Exception as e:
        print(f"CH{ch} Config Error: {e}")

In [4]:
from lakeshore.model_240 import Model240InputParameter, Model240Enums

def reset_channel(device, ch):
    print(f"Resetting CH{ch}...")

    param = Model240InputParameter()

    # ⚠️ Set a safe default: resistance/thermistor-like input
    param.input_type = Model240Enums.InputType.RESISTANCE
    param.autorange = True
    param.range = Model240Enums.Range.AUTO
    param.units = Model240Enums.InputUnits.KELVIN  # or SENSOR if available

    try:
        device.set_input_parameter(ch, param)
        time.sleep(0.5)

        # Force read to refresh state
        device.query(f"SRDG? {ch}")

        print(f"CH{ch} reset done.")
    except Exception as e:
        print(f"Reset failed CH{ch}: {e}")

In [5]:
from sqlalchemy import true
from torch import i0


def main():
    from lakeshore import Model240

    PORT = "COM4"

    device = Model240(com_port=PORT)

    # Step 1: scan everything
    scan_all_channels(device)

    # print("\n--- Configs ---")
    # for ch in [1, 3, 5]:
    #     inspect_config(device, ch)

    # Step 2: reset problematic channels
    # print("\n--- Resetting CH1 & CH3 ---")
    # reset_channel(device, 1)
    # reset_channel(device, 3)

    # time.sleep(1)

    # Step 3: re-check
    print("\n--- After Reset ---")
    # inspect_channel(device, 1)
    # inspect_channel(device, 3)
    # inspect_channel(device, 5)


    channel = 5
    device.get_curve_header(channel)
    device.get_input_parameter(channel)
    for i in range(3):
        try:
            device.get_curve_data_point(channel, i)
        except:
            pass


if __name__ == "__main__":
    main()

=== Channel Scan ===
CH1: Temp = +0.00000, Sensor = +00.0000
CH2: Temp = +0.00000, Sensor = +000.000
CH3: Temp = +0.00000, Sensor = +00.0000
CH4: Temp = +0.00000, Sensor = +0001.65
CH5: Temp = +0297.81, Sensor = +0109.59
CH6: Temp = +0.00000, Sensor = +02.2066
CH7: Temp = +0.00000, Sensor = +00.0000
CH8: Temp = +0.00000, Sensor = +00.0000

--- After Reset ---
